# A.1: Lista Roja de Ecosistemas (Red List of Ecosystems)

In [46]:
import datetime
now = datetime.datetime.now()
print(f"Última actualización de este notebook: {now.strftime('%d-%m-%Y %H:%M')}")

Última actualización de este notebook: 23-03-2026 18:45


El Indicador de Cabecera A.1, denominado "Lista Roja de Ecosistemas" (RLE, por sus siglas en inglés), evalúa el riesgo de colapso de los ecosistemas a nivel nacional. Se expresa a través del Índice de la Lista Roja de Ecosistemas (RLIe), una métrica que cuantifica qué tan cerca están los ecosistemas de perder su distribución natural o su función ecológica fundamental. El índice se mide en una escala decimal de 0 a 1, donde un valor de 1 indica que todos los ecosistemas se encuentran en la categoría de "Preocupación Menor" (intactos o sin riesgo inminente), y un valor de 0 indica que todos los ecosistemas evaluados han "Colapsado". Este indicador es fundamental para priorizar esfuerzos de conservación y monitorear la salud estructural y funcional del capital natural del país.

Los metadatos oficiales del indicador se encuentran disponibles [aquí](https://www.gbf-indicators.org/metadata/headline/A-1).
## Metodología de cálculo
El cálculo del Índice (RLIe) sigue el estándar global de la UICN (https://iucnrle.org/). La unidad de evaluación a nivel nacional corresponde a los Pisos Vegetacionales (Luebert & Pliscoff, 2006), los cuales son agregados posteriormente a los Grupos Funcionales de Ecosistemas (EFG, Nivel 3 de la Tipología Global de la UICN) (Keith et al., 2020) para su reporte internacional.

La metodología consiste en asignar un "peso" numérico a cada categoría de riesgo de la Lista Roja obtenida por cada piso vegetacional: Preocupación Menor (LC) = 0, Casi Amenazado (NT) = 1, Vulnerable (VU) = 2, En Peligro (EN) = 3, En Peligro Crítico (CR) = 4, y Colapsado (CO) = 5.

El RLE index se calcula utilizando la fórmula de agregación: 1 - (Suma de los pesos de todos los ecosistemas evaluados / (Número total de ecosistemas evaluados × Peso máximo posible [5])). Este cálculo se realiza primero para obtener el valor nacional agregado (territorio continental) y luego se desagrega filtrando y calculando el índice para cada EFG de forma independiente.

## Fuentes de datos
Para la construcción y evaluación de este indicador, se utilizó como fuente de información primaria la Capa de Pisos Vegetacionales de Chile (Luebert y Pliscoff, 2015; actualizada al año 2017). Esta cartografía cuenta con una clasificación oficial estructurada de acuerdo con las directrices de la Unión Internacional para la Conservación de la Naturaleza (UICN) para evaluar el riesgo de colapso de los ecosistemas terrestres a nivel nacional.

Los datos, tanto en su formato geoespacial como tabular, fueron descargados directamente desde la plataforma del Sistema de Información de Biodiversidad (SIMBIO) administrada por el Ministerio del Medio Ambiente: https://simbio.mma.gob.cl/Ecosistemas 

Cabe destacar una precisión metodológica: de acuerdo con la revisión de la planilla de datos descargada desde SIMBIO, para la determinación del criterio final UICN (el cual define la categoría de amenaza y es el insumo principal utilizado para el cálculo de este indicador), se evaluaron específicamente los Criterios A2 (reducción de la distribución geográfica en los últimos 50 años) y A3 (reducción histórica de la distribución geográfica) de la metodología global de la Lista Roja de Ecosistemas.

Elaboración de la Capa de EFG para Chile: Para la homologación espacial hacia los Grupos Funcionales de Ecosistemas (EFG) de la UICN, se realizó un esfuerzo técnico de integración geomática que unió cuatro fuentes cartográficas oficiales: la clasificación de Pisos Vegetacionales de Chile (Luebert & Pliscoff, 2006), la cartografía de Ecosistemas Marinos (Rovira & Herreros, 2016), el Inventario Nacional de Humedales y el Inventario Nacional de Glaciares.

In [6]:
URL = 'https://simbio.mma.gob.cl/Ecosistemas/GetExcelExtendido'

In [7]:
import io
import re
from datetime import datetime

import numpy as np
import pandas as pd
import requests
from IPython.display import Markdown, display

# -------------------------
# Configuración / constantes
# -------------------------

#SHEET_ID = "1WQ5eKmttF25JonqEcwyKcd28PBfmnb16"
SHEET_NAME = "General"

#URL = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=xlsx"


weight_category_map = {
    "DD": 0,
    "NE": 0,
    "LC": 0,
    "NT": 1,
    "VU": 2,
    "EN": 3,
    "CR": 4,
    "CO": 5,
}

M = 5  # Peso máximo para el cálculo del RLI

def compute_rli(group: pd.DataFrame) -> pd.Series:
    """Calcula el RLI sobre un grupo de filas (año, o global)."""
    N = len(group)
    sum_w = group["weight_category"].sum()
    rli = 1 - (sum_w / (N * M)) if N > 0 else np.nan
    return pd.Series(
        {
            "Número total de ecosistemas (pisos vegetacionales) evaluados.": N,
            "suma_pesos": sum_w,
            "RLI": rli,
        }
    )


# -------------------------
# Descarga y preparación del DataFrame
# -------------------------

resp = requests.get(URL)
resp.raise_for_status()

df = pd.read_excel(
    io.BytesIO(resp.content),
    sheet_name=SHEET_NAME,
)

df["criterio_final_uicn"] = df["Criterio final de UICN"]

# -------------------------
# Codificar categorías y pesos
# -------------------------
# Excluir DD y R
df = df[~df["criterio_final_uicn"].isin([""])]
df["weight_category"] = df["criterio_final_uicn"].map(weight_category_map).astype("Int64")
# Eliminar filas sin categoría o sin peso
df = df.dropna(subset=["criterio_final_uicn", "weight_category"])

# -------------------------
# Cálculo del RLI
# -------------------------

rli_global = compute_rli(df)
print("RLI global:")
print(rli_global)

RLI global:
Número total de ecosistemas (pisos vegetacionales) evaluados.    125.0000
suma_pesos                                                        64.0000
RLI                                                                0.8976
dtype: float64


In [43]:
lc_count=len(df[df['criterio_final_uicn']=='LC'])
rli_ecosystem_count=rli_global.iloc[0].round(0)
rli_weight=rli_global.iloc[1]
rli_result=rli_global.iloc[2]

## Resultados

In [44]:
from IPython.display import Markdown as md
text = f""" 
Para el actual ciclo de reporte, el Índice de la Lista Roja de Ecosistemas (RLIe) para el territorio continental de Chile alcanzó un valor nacional agregado de {rli_result}. Este resultado indica que, a nivel macro, los ecosistemas terrestres del país enfrentan un riesgo relativamente bajo de colapso ecológico, traccionado por una amplia base de ecosistemas que aún mantienen su integridad. Del total de {rli_ecosystem_count} pisos vegetacionales evaluados en el territorio continental, {lc_count} se encuentran en la categoría de "Preocupación Menor" (LC). Sin embargo, el análisis desagregado revela ecosistemas específicos sometidos a severas presiones.
"""
md(text)

 
Para el actual ciclo de reporte, el Índice de la Lista Roja de Ecosistemas (RLIe) para el territorio continental de Chile alcanzó un valor nacional agregado de 0.8976. Este resultado indica que, a nivel macro, los ecosistemas terrestres del país enfrentan un riesgo relativamente bajo de colapso ecológico, traccionado por una amplia base de ecosistemas que aún mantienen su integridad. Del total de 125.0 pisos vegetacionales evaluados en el territorio continental, 106 se encuentran en la categoría de "Preocupación Menor" (LC). Sin embargo, el análisis desagregado revela ecosistemas específicos sometidos a severas presiones.


In [14]:
#df.to_excel("A_1_rli_ecosystem.xlsx")

In [15]:
import geopandas as gpd
from sqlalchemy import create_engine
import os
def make_engine_from_env(host_override=None):
    host = host_override or os.getenv("POSTGRES_HOST", "localhost")
    port = os.getenv("POSTGRES_PORT", "5432")
    user = os.getenv("POSTGRES_USER", "postgres")
    password = 'super_secret_pass'#os.getenv("POSTGRES_PASSWORD", "")
    db = os.getenv("POSTGRES_DB", "postgres")
    password = password.strip("'").strip('"')
    url = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{db}"
    return create_engine(url, pool_pre_ping=True)

engine = make_engine_from_env(host_override="db")  # desde tu host
efg_continental_gpd = gpd.read_postgis(
    sql="SELECT * FROM public.efg_continental",
    con=engine,
    geom_col="geometry",     
    crs="EPSG:4326"      
)

DETAIL:  The database was created using collation version 2.41, but the operating system provides version 2.31.
HINT:  Rebuild all objects in this database that use the default collation and run ALTER DATABASE postgres REFRESH COLLATION VERSION, or build PostgreSQL with the right library version.


In [16]:
efg_continental_df = pd.DataFrame(efg_continental_gpd)

In [17]:
mask = df["Nombre"] == "Bosque esclerofilo mediterraneo andino de Lithrea caustica - Lomatia hirsuta"
df.loc[mask, "Nombre"] = "Bosque esclerofilo mediterráneo andino de Lithrea caustica - Lomatia hirsuta"

mask = df["Nombre"] == "Bosque esclerofilo psamófilo mediterráneo interior de Quillaja saponaria/Fabiana imbricata"
df.loc[mask, "Nombre"] = "Bosque esclerofilo psamófilo mediterráneo interior de Quillaja saponaria / Fabiana imbricata"

mask = df["Nombre"] == "Bosque caducifolio mediterráneo costero de Nothofagus glauca y Persea lingue"
df.loc[mask, "Nombre"] = "Bosque caducifolio mediterráneo costero de Nothofagus glauca - Persea lingue"

mask = df["Nombre"] == "Bosque caducifolio mediterráneo andino de Nothofagus glauca - Nothofagus obliqua"
df.loc[mask, "Nombre"] = "Bosque caducifolio mediterráneo andino de Nothofagus glauca - N. obliqua"

mask = df["Nombre"] == "Bosque caducifolio mediterráneo-templado costero de Nothofagus obliqua - Gomortega keule"
df.loc[mask, "Nombre"] = "Bosque caducifolio mediterráneo costero de Nothofagus obliqua - Gomortega keule"

mask = df["Nombre"] == "Bosque caducifolio templado de Nothofagus obliqua - Persea lingue"
df.loc[mask, "Nombre"] = "Bosque caducifolio mediterráneo de Nothofagus obliqua - Persea lingue"

mask = df["Nombre"] == "Bosque mixto templado costero de Nothofagus dombeyi - Nothofagus obliqua"
df.loc[mask, "Nombre"] = "Bosque mixto mediterráneo-templado costero de Nothofagus dombeyi - N. obliqua"

mask = df["Nombre"] == "Herbazal efímero tropical costero de Nolana adansonii - Nolana lycioides"
df.loc[mask, "Nombre"] = "Herbazal efímero tropical costero de Nolana adansonii - N. lycioides"

mask = df["Nombre"] == "Matorral desértico tropical interior de Malesherbia auristipulata - Tarasa operculata"
df.loc[mask, "Nombre"] = "Matorral desértico tropical interior Malesherbia auristipulata - Tarasa operculata"

mask = df["Nombre"] == "Matorral desértico mediterráneo costero de Copiapoa boliviana - Heliotropium pycnophyllum"
df.loc[mask, "Nombre"] = "Matorral desértico tropical-mediterráneo costero de Copiapoa boliviana - Heliotropium pycnophyllum"

mask = df["Nombre"] == "Matorral desértico mediterráneo costero de Euphorbia lactiflua y Eulychnia iquiquensis"
df.loc[mask, "Nombre"] = "Matorral desértico mediterráneo costero de Euphorbia lactiflua / Eulychnia iquiquensis"

mask = df["Nombre"] == "Matorral desértico mediterráneo interior de Oxyphyllum ulicinum y Gymnophyton foliosum"
df.loc[mask, "Nombre"] = "Matorral desértico mediterráneo interior de Oxyphyllum ulicinum - Gymnophyton foliosum"

mask = df["Nombre"] == "Matorral desértico mediterráneo interior de Skytanthus acutus y Atriplex deserticola"
df.loc[mask, "Nombre"] = "Matorral desértico mediterráneo interior de Skytanthus acutus - Atriplex deserticola"

mask = df["Nombre"] == "Matorral desértico tropical interior de Huidobria chilensis y Nolana leptophylla"
df.loc[mask, "Nombre"] = "Matorral desértico tropical interior de Huidobria chilensis - Nolana leptophylla"

mask = df["Nombre"] == "Matorral desértico mediterráneo costero de Oxalis virgosa y Heliotropium stenophyllum"
df.loc[mask, "Nombre"] = "Matorral desértico mediterráneo costero de Oxalis virgosa - Heliotropium stenophyllum"

mask = df["Nombre"] == "Matorral desértico mediterráneo interior de Adesmia argentea y Bulnesia chilensis"
df.loc[mask, "Nombre"] = "Matorral desértico mediterráneo interior de Adesmia argentea - Bulnesia chilensis"

mask = df["Nombre"] == "Matorral desértico mediterráneo interior de Heliotropium stenophyllum y Flourensia thurifera"
df.loc[mask, "Nombre"] = "Matorral desértico mediterráneo interior de Heliotropium stenophyllum - Flourensia thurifera"

mask = df["Nombre"] == "Matorral desértico mediterráneo interior de Flourensia thurifera y Colliguaja odorifera"
df.loc[mask, "Nombre"] = "Matorral desértico mediterráneo interior de Flourensia thurifera - Colliguaja odorifera"

mask = df["Nombre"] == "Bosque espinoso tropical interior de Prosopis tamarugo / Tessaria absinthioides"
df.loc[mask, "Nombre"] = "Bosque espinoso tropical interior de Prosopis tamarugo / Tessaria absinthiodes"

mask = df["Nombre"] == "Matorral arborescente esclerofilo mediterráneo interior de Quillaja saponaria / Porlieria chilensis"
df.loc[mask, "Nombre"] = "Matorral arborescente esclerofilo mediterráneo interior Quillaja saponaria / Porlieria chilensis"

mask = df["Nombre"] == "Bosque esclerofilo mediterráneo andino de Kageneckia angustifolia - Guindilia trinervis"
df.loc[mask, "Nombre"] = "Bosque esclerofilo mediterráneo andino de Kageneckia angustifolia / Guindilia trinervis"

mask = df["Nombre"] == "Bosque caducifolio mediterráneo andino de Nothofagus obliqua - Austrocedrus chilensis"
df.loc[mask, "Nombre"] = "Bosque caducifolio mediterráneo-templado andino de Nothofagus obliqua - Austrocedrus chilensis"

mask = df["Nombre"] == "Bosque caducifolio mediterráneo-templado andino de Nothofagus alpina - Nothofagus obliqua"
df.loc[mask, "Nombre"] = "Bosque caducifolio mediterráneo-templado andino de Nothofagus alpina - N. obliqua"

mask = df["Nombre"] == "Bosque caducifolio templado andino de Nothofagus alpina - Nothofagus dombeyi"
df.loc[mask, "Nombre"] = "Bosque caducifolio templado andino de Nothofagus alpina - N. dombeyi"

mask = df["Nombre"] == "Bosque caducifolio mediterráneo-templado andino de Nothofagus pumilio - Nothofagus obliqua"
df.loc[mask, "Nombre"] = "Bosque caducifolio mediterráneo-templado andino de Nothofagus pumilio - N. obliqua"

mask = df["Nombre"] == "Matorral arborescente caducifolio mediterráneo-templado oriental de Nothofagus antárctica / Berberis microphylla"
df.loc[mask, "Nombre"] = "Matorral arborescente caducifolio mediterráneo-templado oriental de Nothofagus antarctica / Berberis microphylla"

mask = df["Nombre"] == "Bosque resinoso templado andino de Araucaria araucana / Festuca scabriuscula"
df.loc[mask, "Nombre"] = "Bosque resinoso mediterráneo-templado andino de Araucaria araucana / Festuca scabriuscula"

mask = df["Nombre"] == "Bosque resinoso templado costero de Pilgerodendron uviferum y Tepualia stipularis"
df.loc[mask, "Nombre"] = "Bosque resinoso templado costero de Pilgerodendron uvifera - Tepualia stipularis"

mask = df["Nombre"] == "Bosque resinoso templado costero de Pilgerodendron uviferum / Astelia pumila"
df.loc[mask, "Nombre"] = "Bosque resinoso templado costero de Pilgerodendron uvifera / Astelia pumila"

mask = df["Nombre"] == 'Bosque siempreverde mixto templado andino de Nothofagus betuloides / Berberis ilicifolia"'
df.loc[mask, "Nombre"] = "Bosque mixto templado andino de Nothofagus betuloides / Berberis ilicifolia"

mask = df["Nombre"] == "Bosque mixto templado-antiboreal andino de Nothofagus betuloides / Nothofagus pumilio"
df.loc[mask, "Nombre"] = "Bosque mixto templado-antiboreal andino de Nothofagus betuloides - Nothofagus pumilio"

mask = df["Nombre"] == "Bosque siempreverde templado-antiboreal costero de Nothofagus betuloides - Embothrium coccineum"
df.loc[mask, "Nombre"] = "Bosque siempreverde antiboreal costero de Nothofagus betuloides - Embothrium coccineum"

mask = df["Nombre"] == "Matorral siempreverde templado costero de Pilgerodendron uviferum -Nothofagus nitida"
df.loc[mask, "Nombre"] = "Matorral siempreverde templado costero de Pilgerodendron uvifera - Nothofagus nitida"

mask = df["Nombre"] == "Turbera antiboreal costera de Bolax bovei - Phyllachne uliginosa"
df.loc[mask, "Nombre"] = "Turbera templada-antiboreal costera de Bolax caespitosus - Phyllachne uliginosa"

mask = df["Nombre"] == "Matorral bajo tropical andino de Parastrephia lepidophylla - Parastrephia quadrangularis"
df.loc[mask, "Nombre"] = "Matorral bajo tropical andino de Parastrephia lepidophylla ? P. qudrangularis"

mask = df["Nombre"] == "Matorral bajo tropical andino de Adesmia frigida / Stipa frigida"
df.loc[mask, "Nombre"] = "Matorral bajo tropical andino de Adesmia frigida / Jarava frigida"

mask = df["Nombre"] == "Matorral bajo templado andino de Adesmia longipes - Senecio bipontini"
df.loc[mask, "Nombre"] = "Matorral bajo templado andino de Adesmia longipes - Senecio bipontinii"

mask = df["Nombre"] == "Matorral bajo templado-antiboreal andino de Bolax gummifera - Azorella selago"
df.loc[mask, "Nombre"] = "Matorral bajo antiboreal andino de Bolax gummifera - Azorella selago"

mask = df["Nombre"] == "Herbazal tropical andino de Chaetanthera sphaeroidalis"
df.loc[mask, "Nombre"] = "Herbazal tropical-mediterráneo andino de Chaetanthera sphaeroidalis"

mask = df["Nombre"] == "Herbazal antiboreal andino de Nassauvia pygmaea – N. lagascae"
df.loc[mask, "Nombre"] = "Herbazal antiboreal andino de Nassauvia pygmaea - N. lagascae"

mask = df["Nombre"] == "Estepa mediterránea-templada oriental de Festuca gracillima"
df.loc[mask, "Nombre"] = "Estepa mediterránea oriental de Festuca gracillima"

mask = df["Nombre"] == "Estepa mediterránea-templada oriental de Festuca gracillima / Mulinum spinosum"
df.loc[mask, "Nombre"] = "Estepa mediterránea oriental de Festuca gracillima / Mulinum spinosum"


In [18]:
df_join = df.merge(
    efg_continental_df,
    how="left",
    left_on="Nombre",      # columnas en df1
    right_on="PISO"      # columnas en df2 (ajusta si tienen otro nombre)
)

In [19]:
#df_join.to_excel("A_1_rli_ecosystem_join_efg_continental.xlsx")

In [20]:
df_join["FORMACION"] = df_join["FORMACION"].astype(str).str.strip()

rli_por_formacion = (
    df_join.groupby("FORMACION", dropna=False)
      .apply(compute_rli)
      .reset_index()
      .sort_values("RLI", ascending=True)  # o False si quieres mejores primero
)

In [26]:
tabla_md = (
    rli_por_formacion
    .loc[:, ["FORMACION", "RLI"]]
    .sort_values("RLI", ascending=True)
)
tabla_md["RLI"] = tabla_md["RLI"].round(4)
md = tabla_md.to_markdown(index=False)
display(Markdown(md))

| FORMACION                |    RLI |
|:-------------------------|-------:|
| Bosque caducifolio       | 0.6476 |
| Bosque esclerofilo       | 0.65   |
| Bosque laurifolio        | 0.7333 |
| Bosque espinoso          | 0.7429 |
| Bosque resinoso          | 1      |
| Bosque siempreverde      | 1      |
| Desierto absoluto        | 1      |
| Dunas de aerófitos       | 1      |
| Estepas y pastizales     | 1      |
| Matorral siempreverde    | 1      |
| Matorral bajo de altitud | 1      |
| Herbazal efímero         | 1      |
| Matorral bajo desértico  | 1      |
| Matorral caducifolio     | 1      |
| Matorral desértico       | 1      |
| Matorral esclerofilo     | 1      |
| Matorral espinoso        | 1      |
| Herbazal de altitud      | 1      |
| Turberas                 | 1      |

In [24]:
df_join["Group_"] = df_join["Group_"].astype(str).str.strip()

rli_por_piso = (
    df_join.groupby("Group_", dropna=False)
      .apply(compute_rli)
      .reset_index()
      .sort_values("RLI", ascending=True)  # o False si quieres mejores primero
)

In [27]:
tabla_md = (
    rli_por_piso
    .loc[:, ["Group_", "RLI"]]
    .sort_values("RLI", ascending=True)
)
tabla_md["RLI"] = tabla_md["RLI"].round(4)
md = tabla_md.to_markdown(index=False)
display(Markdown(md))

| Group_                                        |    RLI |
|:----------------------------------------------|-------:|
| Seasonally dry temperate heath and shrublands | 0.72   |
| Tropical/Subtropical dry forests and thickets | 0.8    |
| Oceanic cool temperate rainforests            | 0.8217 |
| Boreal, temperate and montane peat bogs       | 1      |
| Hyper-arid deserts                            | 1      |
| Muddy Shorelines                              | 1      |
| Semi-desert steppe                            | 1      |
| Succulent or Thorny deserts and semi-deserts  | 1      |
| Temperate alpine grasslands and shrublands    | 1      |
| Temperate subhumid grasslands                 | 1      |
| Tropical alpine grasslands and herbfields     | 1      |

### Desagregación por Grupos Funcionales:

Al analizar el índice a nivel de Grupos Funcionales de Ecosistemas (EFG), se evidencian variaciones críticas asociadas principalmente a la zona central y centro-sur del país (clima mediterráneo y templado):

* Matorrales y brezales templados estacionalmente secos (T3.2 Seasonally dry temperate heath and shrublands): Es el grupo funcional que presenta el índice más bajo y crítico a nivel nacional, con un valor de 0.7200. De los 10 pisos vegetacionales que conforman este EFG, 3 se encuentran clasificados en estado de "En Peligro Crítico" (CR), destacando la frágil situación del Bosque esclerófilo mediterráneo costero y el Bosque esclerófilo mediterráneo interior.
* Bosques y matorrales secos tropicales/subtropicales (T1.2 Tropical/Subtropical dry forests and thickets): Presenta un índice de 0.8000. Este valor se explica porque, de sus 9 pisos vegetacionales, uno se encuentra "En Peligro" (EN) —el Bosque espinoso mediterráneo interior de Acacia caven— y otros tres están categorizados como "Vulnerables" (VU).
* Bosques húmedos templados fríos oceánicos (T2.3 Oceanic cool temperate rainforests): Este grupo funcional, el más extenso y diverso del país, reporta un índice de 0.8217. Aunque concentra 35 pisos en Preocupación Menor (LC) hacia el sur austral, la presión sobre su límite norte genera que 9 pisos vegetacionales estén categorizados "En Peligro Crítico" (CR), asociados en su totalidad a los bosques caducifolios y laurifolios dominados por especies de Nothofagus (p. ej., N. glauca) en la zona costera y precordillera central.

### Ecosistemas Íntegros (Índice Máximo) 
En marcado contraste con la zona central, existen seis Grupos Funcionales de Ecosistemas que mantienen un RLE de 1.0000, lo que indica que el 100% de los pisos vegetacionales que los componen se encuentran en la categoría de "Preocupación Menor" (LC). Esta integridad ecológica se concentra fundamentalmente en biomas extremos, áridos o de alta montaña, tales como los Desiertos hiperaridos (T5.5 Hyper-arid deserts), las Estepas semidesérticas (T5.1 Semi-desert steppe), los Desiertos suculentos y espinosos (T5.2 Succulent or Thorny deserts and semi-deserts), y los Herbazales alpinos tropicales y templados (T6.4 Temperate alpine grasslands and shrublands y T6.5 Tropical alpine grasslands and herbfields). Asimismo, destacan por su buen estado de conservación los sistemas ecotonales del sur profundo, representados por las Turberas boreales, templadas y montañas (TF1.6 Boreal, temperate and montane peat bogs).


## Observaciones
Al contrastar la base de datos espacial descargada desde SIMBIO con la literatura oficial que sustenta la evaluación, se evidencia una discrepancia en el reporte de la información. Mientras que la planilla de SIMBIO indica que el criterio final de riesgo se sustentó exclusivamente en los Criterios A2 y A3 (reducción espacial e histórica de la distribución geográfica), el documento oficial del estudio (Informe Final: Evaluación del estado de conservación de los ecosistemas terrestres de Chile, MMA, 2015) detalla que la evaluación original también aplicó y consideró los Criterios B2 (distribución restringida) y C2 (degradación ambiental). Esta diferencia entre el reporte técnico y la plantilla descargada del portal SIMBIO evidencia la necesidad de armonizar las bases de datos públicas para reflejar íntegramente el esfuerzo analítico multidimensional realizado por el país.

### Brechas
El análisis de este indicador revela limitaciones críticas tanto en la disponibilidad temporal de los datos como en su cobertura geográfica, las cuales deben ser abordadas para los próximos ciclos de reportes nacionales:
1. **Imposibilidad de analizar tendencias temporales** : La naturaleza de un índice como el RLE radica en su capacidad para medir cambios en el riesgo de colapso a lo largo del tiempo. Dado que Chile cuenta hasta ahora con una única evaluación oficial a nivel nacional (línea base), resulta imposible evaluar y analizar tendencias en el estado de conservación de los ecosistemas.
2. **Desafío de comparabilidad metodológica**: El país enfrenta el desafío de realizar una segunda evaluación nacional que permita habilitar el cálculo de tendencias. Será fundamental para este próximo ejercicio asegurar la comparabilidad estadística con la línea base, lo cual exigirá integrar formalmente y visibilizar en las plataformas oficiales un espectro más amplio de criterios de evaluación de la UICN (incorporando explícitamente y sin ambigüedades los criterios B y C de degradación funcional y distribución, además del criterio A espacial).
3. **Obsolescencia temporal de la evaluación base**: La clasificación del estado de conservación de los ecosistemas terrestres (pisos vegetacionales) fue realizada en el año 2015. Desde entonces, el país ha experimentado eventos transformadores significativos, como megaincendios forestales, la prolongación de la megasequía y cambios intensivos en el uso del suelo, por lo que el nivel de riesgo actual de colapso de varios ecosistemas probablemente sea mayor al reportado. Es prioritario actualizar esta evaluación territorial.
4. **Vacío de información en el ámbito marino**: Actualmente, Chile no cuenta con una evaluación oficial del riesgo de colapso bajo el estándar de la Lista Roja de Ecosistemas de la UICN para su territorio marítimo. Si bien se cuenta con la cartografía base (Ecosistemas Marinos, Rovira y Herreros, 2016) utilizada para reportar extensión, la falta de una categorización de riesgo impide calcular el RLE para los ecosistemas estrictamente marinos, limitando la representatividad nacional del indicador ante el CBD.